# Point Cloud Registration: FPFH + RANSAC + ICP

Registration aligns two or more point clouds into a common coordinate frame.
Occulus provides a complete two-stage pipeline:

1. **Global registration** — FPFH descriptors + RANSAC (no initial guess needed)
2. **Fine registration** — Point-to-plane ICP (refines the coarse estimate)

This is the standard workflow for TLS multi-scan fusion, ALS strip adjustment,
and change detection surveys.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from occulus.types import PointCloud
from occulus.filters import voxel_downsample
from occulus.normals import estimate_normals
from occulus.registration import compute_fpfh, ransac_registration, icp_point_to_plane

## 1. Create Two Misaligned Clouds

We'll simulate a TLS scenario with a known rigid transformation.

In [ ]:
rng = np.random.default_rng(0)

def make_office_scan(rng, n=5000):
    """Generate a synthetic indoor office scan (walls + floor + furniture)."""
    pts = []
    # Floor
    fx, fy = rng.uniform(0, 8, n//3), rng.uniform(0, 6, n//3)
    pts.append(np.column_stack([fx, fy, rng.normal(0, 0.02, n//3)]))
    # Walls
    wx = rng.uniform(0, 8, n//4)
    pts.append(np.column_stack([wx, rng.normal(0, 0.02, n//4), rng.uniform(0, 3, n//4)]))
    pts.append(np.column_stack([wx, rng.normal(6, 0.02, n//4), rng.uniform(0, 3, n//4)]))
    # Desk cluster
    dx, dy = rng.uniform(2, 5, n//6), rng.uniform(2, 4, n//6)
    pts.append(np.column_stack([dx, dy, rng.uniform(0.7, 0.75, n//6)]))
    return np.vstack(pts).astype(np.float64)

# Source cloud (scan position A)
src_xyz = make_office_scan(rng)
source = PointCloud(src_xyz)

# Known transformation: 5° rotation + 0.3 m translation
angle = np.radians(5)
R_true = np.array([[np.cos(angle), -np.sin(angle), 0],
                   [np.sin(angle),  np.cos(angle), 0],
                   [0, 0, 1]])
t_true = np.array([0.3, -0.1, 0.05])
tgt_xyz = (R_true @ src_xyz.T).T + t_true + rng.normal(0, 0.01, src_xyz.shape)
target = PointCloud(tgt_xyz)

print(f'Source: {source.n_points:,} points')
print(f'Target: {target.n_points:,} points')
print(f'True rotation: {np.degrees(angle):.1f}°')
print(f'True translation: {t_true}')

## 2. Preprocessing

In [ ]:
# Downsample + normals
src_ds = voxel_downsample(source, voxel_size=0.1)
tgt_ds = voxel_downsample(target, voxel_size=0.1)

src_n = estimate_normals(src_ds, radius=0.3)
tgt_n = estimate_normals(tgt_ds, radius=0.3)

print(f'After downsample: {src_ds.n_points} / {tgt_ds.n_points} points')

## 3. Global Registration (FPFH + RANSAC)

In [ ]:
import time

# Compute 33-dim FPFH descriptors
src_feat = compute_fpfh(src_n, radius=0.5)
tgt_feat = compute_fpfh(tgt_n, radius=0.5)
print(f'Feature shape: {src_feat.shape}  (N × 33-dim FPFH)')

t0 = time.perf_counter()
coarse = ransac_registration(
    src_n, tgt_n, src_feat, tgt_feat,
    max_correspondence_distance=0.15,
    max_iterations=10_000,
)
print(f'RANSAC — fitness: {coarse.fitness:.4f}, RMSE: {coarse.inlier_rmse:.4f} m  [{time.perf_counter()-t0:.2f}s]')

## 4. Fine Registration (Point-to-Plane ICP)

In [ ]:
t0 = time.perf_counter()
fine = icp_point_to_plane(
    src_n, tgt_n,
    initial_transform=coarse.transformation,
    max_correspondence_distance=0.05,
    max_iterations=100,
)
print(f'ICP   — fitness: {fine.fitness:.4f}, RMSE: {fine.inlier_rmse:.6f} m  [{time.perf_counter()-t0:.2f}s]')
print(f'Converged in {fine.n_iterations} iterations')

# Recover estimated rotation angle
R_est = fine.transformation[:3, :3]
angle_est = np.degrees(np.arctan2(R_est[1, 0], R_est[0, 0]))
t_est = fine.transformation[:3, 3]

print(f'\nEstimated rotation: {angle_est:.3f}° (true: {np.degrees(angle):.3f}°)')
print(f'Estimated translation: {t_est}  (true: {t_true})')
print(f'Rotation error: {abs(angle_est - np.degrees(angle)):.4f}°')
print(f'Translation error: {np.linalg.norm(t_est - t_true):.5f} m')

## 5. Visualise Registration Quality

In [ ]:
# Apply estimated transform to source
T = fine.transformation
src_aligned = (T[:3, :3] @ src_xyz.T).T + T[:3, 3]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(src_xyz[:, 0], src_xyz[:, 1], s=1, c='royalblue', alpha=0.5, label='Source (before)')
axes[0].scatter(tgt_xyz[:, 0], tgt_xyz[:, 1], s=1, c='tomato', alpha=0.5, label='Target')
axes[0].set_title('Before Registration'); axes[0].legend(markerscale=5)
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')

axes[1].scatter(src_aligned[:, 0], src_aligned[:, 1], s=1, c='royalblue', alpha=0.5, label='Source (aligned)')
axes[1].scatter(tgt_xyz[:, 0], tgt_xyz[:, 1], s=1, c='tomato', alpha=0.5, label='Target')
axes[1].set_title(f'After FPFH+RANSAC+ICP  (RMSE={fine.inlier_rmse:.4f} m)')
axes[1].legend(markerscale=5)
axes[1].set_xlabel('X (m)'); axes[1].set_ylabel('Y (m)')

plt.suptitle('Point Cloud Registration: FPFH + RANSAC → ICP', fontsize=13)
plt.tight_layout()
plt.savefig('../../outputs/registration_workflow.png', dpi=150)
plt.show()

## 6. Multi-Scan Chain with `align_scans`

For 3+ overlapping scans, `align_scans` builds a sequential pairwise ICP chain.

In [ ]:
from occulus.registration import align_scans

# Three scans with small incremental offsets
scans = []
for i in range(3):
    offset = np.array([i * 0.2, i * 0.1, 0.0])
    scan_xyz = src_xyz + offset + rng.normal(0, 0.02, src_xyz.shape)
    scans.append(estimate_normals(voxel_downsample(PointCloud(scan_xyz), 0.1), radius=0.3))

alignment = align_scans(scans, max_correspondence_distance=0.2)
print(f'Aligned {len(scans)} scans')
print(f'Global RMSE: {alignment.global_rmse:.5f} m')
for i, res in enumerate(alignment.pairwise_results):
    print(f'  Scan {i}→{i+1}: fitness={res.fitness:.4f}, RMSE={res.inlier_rmse:.5f} m')

**Next**: `05_forest_inventory.ipynb`